In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# --- STEP 1: LOAD DATA ---
# Using the standard URL - if this fails, ensure your Colab has internet access
url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv"
df = pd.read_csv(url)

# --- STEP 2: PREPROCESS DATA ---
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

# Remove leakage and redundant date info
df = df.drop(['reservation_status', 'reservation_status_date', 'arrival_date_year'], axis=1)

X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# --- STEP 3: SPLIT DATA (70/15/15) ---
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1765, random_state=42, stratify=y_train_full)

# --- STEP 4: BUILDING PIPELINES ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# --- STEP 5: TRAIN MODELS ---
# NOTE: SVC is extremely slow on large datasets.
# We will use a smaller sample of the training data just for the SVC fit.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=10),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVC3": SVC(kernel='poly', degree=3, probability=True)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])

    if name == "SVC3":
        # Downsample training data for SVC so it finishes in a few minutes
        X_train_sample, _, y_train_sample, _ = train_test_split(X_train, y_train, train_size=5000, random_state=42, stratify=y_train)
        clf.fit(X_train_sample, y_train_sample)
    else:
        clf.fit(X_train, y_train)

    # Evaluation Helper
    def get_metrics(model_pipe, X_set, y_set):
        preds = model_pipe.predict(X_set)
        probs = model_pipe.predict_proba(X_set)[:, 1]
        return (accuracy_score(y_set, preds),
                precision_score(y_set, preds),
                recall_score(y_set, preds),
                f1_score(y_set, preds),
                roc_auc_score(y_set, probs))

    val_metrics = get_metrics(clf, X_val, y_val)
    test_metrics = get_metrics(clf, X_test, y_test)

    results.append({
        "Model": name,
        "Val_Acc": val_metrics[0], "Val_Prec": val_metrics[1], "Val_Rec": val_metrics[2], "Val_F1": val_metrics[3], "Val_AUC": val_metrics[4],
        "Test_Acc": test_metrics[0], "Test_Prec": test_metrics[1], "Test_Rec": test_metrics[2], "Test_F1": test_metrics[3], "Test_AUC": test_metrics[4]
    })

# --- STEP 6: ENSEMBLE MODELS ---
print("Training Ensembles...")
best_models = [
    ('gbm', GradientBoostingClassifier()),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10)),
    ('dt', DecisionTreeClassifier(max_depth=10))
]

ensembles = {
    "Voting Ensemble": VotingClassifier(estimators=best_models, voting='soft'),
    "Bayesian Stacking": StackingClassifier(estimators=best_models, final_estimator=GaussianNB())
}

for name, ensemble in ensembles.items():
    e_clf = Pipeline(steps=[('preprocessor', preprocessor), ('ensemble', ensemble)])
    e_clf.fit(X_train, y_train)

    val_m = get_metrics(e_clf, X_val, y_val)
    test_m = get_metrics(e_clf, X_test, y_test)

    results.append({
        "Model": name,
        "Val_Acc": val_m[0], "Val_Prec": val_m[1], "Val_Rec": val_m[2], "Val_F1": val_m[3], "Val_AUC": val_m[4],
        "Test_Acc": test_m[0], "Test_Prec": test_m[1], "Test_Rec": test_m[2], "Test_F1": test_m[3], "Test_AUC": test_m[4]
    })

# --- FINAL TABLE ---
results_df = pd.DataFrame(results)
print("\n--- Final Comparison Table ---")
display(results_df) # Use display() for pretty formatting in Colab

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training KNN...
Training SVC3...
Training Ensembles...

--- Final Comparison Table ---


,Model,Val_Acc,Val_Prec,Val_Rec,Val_F1,Val_AUC,Test_Acc,Test_Prec,Test_Rec,Test_F1,Test_AUC
0,Logistic Regression,0.819060,0.818267,0.657573,0.729172,0.900350,0.819755,0.815955,0.662948,0.731537,0.898010
1,Decision Tree,0.837930,0.825541,0.713188,0.765262,0.917732,0.839019,0.825664,0.716762,0.767369,0.918256
2,Random Forest,0.788912,0.961812,0.447928,0.611208,0.919285,0.792730,0.965880,0.456587,0.620061,0.921883
3,Gradient Boosting,0.849654,0.839243,0.734891,0.783608,0.925033,0.849629,0.838051,0.736358,0.783920,0.926574
4,KNN,0.827713,0.780553,0.744084,0.761883,0.893025,0.834497,0.786450,0.759421,0.772699,0.897800
5,SVC3,0.820623,0.859605,0.616428,0.717985,0.890984,0.820705,0.861457,0.614863,0.717565,0.888524
6,Voting Ensemble,0.847309,0.866404,0.694951,0.771264,0.928204,0.845943,0.864672,0.692493,0.769063,0.929942
7,Bayesian Stacking,0.849375,0.818476,0.762472,0.789482,0.928490,0.849461,0.818608,0.762587,0.789605,0.930053
